# 시크릿 에이전트 — 레시피 RAG 실행 노트북 (Colab Pro+ / A100·H100)

단계별로 위에서부터 셀을 실행하세요.
1. 의존성 설치  2. 프로젝트 파일 준비  3. (HF 로그인)  4. 레시피 DB 확인  
5. **검색 스모크 테스트**(가벼움 — RAG ① 검증)  6. **6모델 전체 실험**(서브프로세스 루프, RAG ②③ 포함)  7. 결과 집계

> Pro+ **백그라운드 실행**을 켜두면 6번 셀 실행 후 맥북을 닫아도 끝까지 돕니다.

## 1) 의존성 설치

In [ ]:
!pip install -q -U transformers accelerate sentence-transformers rank_bm25
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2) 프로젝트 파일 준비
이미 작업 폴더에 파일들(`rag_retrieval.py`, `recipe_prompts.py`, `recipe_db.json` 등)이 있으면 이 셀은 건너뛰어도 됩니다.\
없으면 GitHub에서 clone합니다. (private 레포면 토큰 입력, public이면 그냥 Enter)

In [ ]:
import os, getpass, subprocess

NEED = {'rag_retrieval.py', 'recipe_prompts.py', 'story_prompts.py', 'recipe_db.json', 'rag_experiment.py'}
if not NEED.issubset(set(os.listdir('.'))):
    if not os.path.isdir('SSU_term_project-1'):
        tok = getpass.getpass('GitHub token (private이면 입력, public이면 Enter): ').strip()
        auth = f'{tok}@' if tok else ''
        subprocess.run(['git', 'clone', f'https://{auth}github.com/topop0628/SSU_term_project-1.git'], check=True)
    os.chdir('SSU_term_project-1')

print('cwd:', os.getcwd())
print('files:', sorted(f for f in os.listdir('.') if not f.startswith('.')))

## 3) (선택) HuggingFace 로그인
EEVE는 공개 모델이라 보통 없어도 되지만, 다운로드 레이트리밋 방지용으로 권장.

In [ ]:
from huggingface_hub import login
import getpass
login(getpass.getpass('HF token (없으면 Enter): ') or None)

## 4) 레시피 DB 확인 / (없으면) 생성
레포에 `recipe_db.json`(2147개)이 이미 들어있습니다. 없을 때만 다시 만듭니다.

In [ ]:
import os, json
if not os.path.exists('recipe_db.json'):
    !python build_recipe_db.py --limit 2000
db = json.load(open('recipe_db.json'))
carrot = sum(1 for r in db if any('당근' in i for i in r['ingredients']))
print(f'레시피 {len(db)}개 | 당근 포함 {carrot}개')

## 5) 검색 스모크 테스트 (가벼움 — RAG ① 확인)
큰 LLM 없이 임베딩·리랭커(소형)만 받아서 **검색→리랭킹**이 동작하는지 빠르게 확인합니다.\
당근/계란/치즈로 검색했을 때 관련 레시피가 나오면 ① 정상.

In [ ]:
import importlib, rag_retrieval as rag
importlib.reload(rag)

retriever = rag.RecipeRetriever('recipe_db.json')   # 임베딩 색인 구축(1~2분)
docs = retriever.retrieve('당근, 계란, 치즈', '당근', top_n=3, use_rerank=True)
for i, d in enumerate(docs, 1):
    print(f"[{i}] {d['dish_name']}  | 재료: {d['ingredients'][:6]}")
    print('    단계1:', d['steps'][0][:50], '...')

## 6) 전체 실험 — 6모델 × (Baseline vs RAG) + 결과 자동 백업
- **모델마다 별도 서브프로세스**로 실행 → 종료 시 OS가 VRAM 100% 회수(런타임 재시작 불필요).
- **모델 1개 끝날 때마다 `rag_results`를 GitHub에 push** → 와이파이/런타임이 끊겨도 끝난 건 안전.
- **이미 완료된 모델은 스킵** → 끊긴 뒤 재실행하면 남은 것만 이어서 돎.

> 실행하면 push용 **GitHub 토큰**(repo 권한)을 한 번 물어봅니다.
> 재시작했다면 셀 1·2(설치·clone)부터 다시 → clone하면 이전 결과도 GitHub에서 같이 받아져 스킵됩니다.

In [ ]:
import sys, subprocess, os, glob, time, getpass

MODELS = [
    'yanolja/EEVE-Korean-Instruct-10.8B-v1.0',
    'rtzr/ko-gemma-2-9b-it',
    'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'trillionlabs/Trillion-7B-preview',
    'allganize/Llama-3-Alpha-Ko-8B-Instruct',
    'Upstage/SOLAR-10.7B-Instruct-v1.0',
]

# --- 결과를 GitHub에 자동 백업 (중간에 끊겨도 끝난 건 안전) ---
subprocess.run(['git', 'config', 'user.email', 'colab@run.local'])
subprocess.run(['git', 'config', 'user.name', 'colab-runner'])
_tok = getpass.getpass('GitHub token (결과 push용, repo 권한): ').strip()
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{_tok}@github.com/topop0628/SSU_term_project-1.git'])

def push_results(msg):
    subprocess.run(['git', 'add', 'rag_results'])
    if subprocess.run(['git', 'commit', '-m', msg]).returncode == 0:
        subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'])
        subprocess.run(['git', 'push', 'origin', 'main'])
        print('  ⬆️ GitHub에 push됨:', msg, flush=True)

# --- 이미 완료된 모델은 스킵 (재시작/재실행 대비) ---
done = {os.path.basename(p).split('rag_ablation_')[1].rsplit('_', 2)[0]
        for p in glob.glob('rag_results/rag_ablation_*.json')}

for i, m in enumerate(MODELS, 1):
    short = m.split('/')[-1]
    if short in done:
        print(f'[{i}/{len(MODELS)}] {short} — 이미 완료, 스킵', flush=True)
        continue
    print(f'\n{"="*70}\n[{i}/{len(MODELS)}] {m}\n{"="*70}', flush=True)
    t0 = time.time()
    # 모델 1개 = 서브프로세스 1개 → 끝나면 OS가 VRAM 완전 회수
    ret = subprocess.run([sys.executable, 'rag_experiment.py', '--model', m])
    print(f'  종료코드 {ret.returncode} | {round((time.time()-t0)/60,1)}분', flush=True)
    if ret.returncode == 0:
        push_results(f'rag 결과: {short}')   # 모델 끝날 때마다 즉시 백업

print('\n🎉 6모델 완료 (결과 GitHub에 백업됨)')

## 7) 결과 확인

In [ ]:
import json, glob, os

# 모델별 JSON 모두 읽어 Baseline vs RAG 요약
rows = {}
for path in sorted(glob.glob('rag_results/rag_ablation_*.json')):
    for r in json.load(open(path)):
        model = r.get('model') or os.path.basename(path)
        rows.setdefault(model, {})[r['condition']] = r

print(f"{'모델':32s} {'요리명(Base)':16s} {'요리명(RAG)':16s} {'근거점수':>6s} {'CRAG':>10s}")
print('-'*90)
for model, conds in rows.items():
    b = conds.get('Baseline', {}).get('recipe', {})
    g = conds.get('RAG', {})
    gr = g.get('recipe', {})
    meta = g.get('rag_meta', {})
    print(f"{model:32s} {str(b.get('dish_name','?'))[:15]:16s} "
          f"{str(gr.get('dish_name','?'))[:15]:16s} "
          f"{str(meta.get('grounding_score','-')):>6s} {str(meta.get('crag_label','-')):>10s}")

# 상세 예시 하나 출력 (첫 모델)
if rows:
    m0 = next(iter(rows))
    print(f"\n--- 상세 예시: {m0} ---")
    for cond in ['Baseline', 'RAG']:
        r = rows[m0].get(cond)
        if r:
            print(f"\n[{cond}] 전략:", r['recipe'].get('strategy'))
            if cond == 'RAG':
                print('  검색:', r['rag_meta'].get('retrieved'))
                print('  통과:', r['rag_meta'].get('kept'))